# Tasty Samplers with chatsnack 🍿

Is popcorn crunchy? Does caramel corn count as dessert? Sometimes we want an answer we can use in an `if` statement.

Let's try a `Sampler`. We give it something to look at and a question, and `ask()` brings back a `Sample` with our answer.

## Got snack?

Install `chatsnack[typesafe]` and add `TYPESAFE_API_KEY` to your `.env` file. Samplers use TypeSafe's Jev model. We'll also use your OpenAI key when we get to the Chat example.

## First bite

Let's start with a bowl of buttered popcorn.

In [ ]:
from chatsnack import Chat, Question, Sampler

sample = Sampler(data="buttered popcorn").ask("Is this crunchy?")
print(sample.answer.yes)


Our answer lives at `sample.answer`. `.yes` gives us a Boolean we can branch on; `.score` gives us the probability of yes, from 0 to 1.

Try swapping the popcorn for melted ice cream. Same question, very different snack.

## Save a little crunch

We could ask about crunch all day. Let's give the question a name and save it for the next snack. The `yes` description lets us say what we mean by crunchy.

Then we'll make a `SnackCheck` Sampler. `{snack}` leaves room for whatever we're eating, and `{question.crunchy}` picks up our saved Question each time we call `ask()`.

In [ ]:
crunchy = Question(
    name="crunchy",
    question="Is this crunchy?",
    yes="It makes a crisp cracking sound when bitten.",
)
crunchy.save()

review = Sampler(name="SnackCheck", data="{snack}", questions=["{question.crunchy}"])
print(review.yaml)
review.save()


### Yummy YAML

Here's our saved `SnackCheck`. Just enough to remember what we're asking:

```yaml
data: "{snack}"
questions:
  - "{question.crunchy}"
```

We can edit the saved YAML in a text editor, just like our chats. Changing the saved `crunchy` Question changes what `SnackCheck` asks next time.

Let's load the Sampler back and give it some popcorn. `.load()` reads the file; `snack=...` fills in the blank.

In [ ]:
review = Sampler(name="SnackCheck")
review.load()
sample = review.ask(snack="popcorn")
print(sample.answer.choice, sample.answer.score)


## The sampler platter

Crunch isn't everything. Let's ask what kind of food we've got and how sweet it is, all in one call.

`choices` gives us options to pick from. `levels` gives us an ordered scale. We'll name the questions so it's easy to find each answer afterward.

In [ ]:
category = Question(name="category", question="What kind of food is this?", choices=["snack", "meal"])
sweetness = Question(name="sweetness", question="How sweet is this?", levels=["Not sweet", "Very sweet"])

sample = review.ask(questions=["{question.crunchy}", category, sweetness], snack="popcorn")
print(sample.answers["category"].choice)
print(sample.answers["sweetness"].score)


`sample.answers["category"]` still finds the category if we shuffle the questions around. Handy once our platter gets bigger.

`sample.answer` always means the first answer, even when we ask several questions. Here that's crunchiness. For sweetness, `.choice` gives us the most likely level and `.score` gives us the model's weighted score.

## Does the menu match the price?

A three-dollar snack and a sixteen-dollar treat probably deserve different introductions. Let's give our menu writer some help deciding how to pitch them.

Here's a tiny pretend menu. Prices are in US dollars, and each item is one serving at a casual snack cafe. Our Sampler will weigh the price against the serving and ingredients, then call it expensive or inexpensive for that setting.

In [ ]:
products = [
    dict(name="Movie popcorn", price_usd=3,
         details="One small bowl of freshly popped corn with butter and salt."),
    dict(name="Truffle popcorn", price_usd=16,
         details="One small bowl of popcorn with truffle butter and shaved Parmesan."),
    dict(name="Cinnamon pretzel", price_usd=4,
         details="One warm soft pretzel rolled in cinnamon sugar."),
]


In [ ]:
price_check = Sampler(
    name="SnackPrices",
    questions=[
        Question(
            name="price",
            question=(
                "Is this single-serving snack expensive or inexpensive for a casual US cafe? "
                "Consider price_usd, portion size, and ingredients."
            ),
            choices={
                "inexpensive": "An affordable everyday snack for this setting.",
                "expensive": "A premium-priced treat or splurge for this setting.",
            },
        ),
    ],
)
print(price_check.yaml)
price_check.save()


Now for the writing. We'll pass each price rating to a Chat and ask for one line of menu copy. An inexpensive snack should sound like an easy everyday pick; an expensive one should sound like a treat worth considering.

Then a second Sampler gets to judge the copy. It checks whether the wording fits the original rating. The rating stays fixed for this check; we're asking how well the writer followed it.

In [ ]:
menu_writer = Chat(
    "Write one line of menu copy for a casual US snack cafe. "
    "For inexpensive items, convey everyday value. For expensive items, convey a premium treat. "
    "Use only the supplied product details. Convey the positioning through the wording; "
    "leave out the numeric price and the literal labels expensive and inexpensive."
).user("{name}: {details} Price: ${price_usd}. Original price rating: {rating}.")

menu_judge = Sampler(
    name="MenuJudge",
    questions=[
        Question(
            name="fits",
            question="Does the description convey the supplied price rating appropriately for this product?",
            yes=(
                "An inexpensive item sounds approachable and good value, or an expensive item "
                "sounds like a premium treat. The wording stays grounded in the product details."
            ),
            no=(
                "The wording suggests the opposite price tier, gives no sense of the intended "
                "positioning, or invents product details to justify it."
            ),
        ),
    ],
)
menu_judge.save()


Let's run the three snacks through and see how our writer does. We keep each original rating alongside the description so we can inspect a disagreement.

In [ ]:
menu = []
for product in products:
    rating = price_check.ask(data=product).answer.choice
    description = menu_writer.ask(**product, rating=rating)
    verdict = menu_judge.ask(data=dict(
        product=product, rating=rating, description=description,
    )).answer
    menu.append(dict(product=product, rating=rating, description=description, verdict=verdict))

    print(f"{product['name']} (${product['price_usd']}): {rating}")
    print(description)
    print("Judge:", "fits" if verdict.yes else "needs another draft", "\n")


Did the writer make the truffle popcorn sound worth a splurge? Did the ordinary popcorn keep its everyday appeal? A judge can disagree, and we can look at its row in `menu` to see which description needs work.

Try changing a price or the writer's instructions and run the loop again. Each item gets one price check, one Chat response, and one review of that response.

## Save some for later

Let's return to our three-question popcorn sample from the sampler platter. Want to try that same bowl again later? `from_sample()` makes a Sampler from the data and questions in that sample. It also keeps the model version that answered.

We can keep tinkering with our saved `crunchy` Question without changing this copy. Running it again may give different answers, but we'll be asking about the same popcorn.

In [ ]:
replay = Sampler.from_sample(sample, name="PopcornReview")
replay.save()
repeated = replay.ask()
print(repeated.answers["category"].choice)
print(repeated.model, repeated.usage)
